# Lesson 07 — Seeing Your Data With Plots

**What this notebook does:** builds a 12-ticket support table (extending Lesson 06's DataFrame), then looks at it four different ways using `matplotlib` — a histogram, a bar chart, a scatter plot, and a box plot — to find things a printed table of numbers hides.

## Step 1 — Why look before modeling, and the plotting library

A table of numbers is honest, but it does not *show* anything — a person scanning 12 rows of `satisfaction_rating` will not reliably notice "shipping tickets swing between very angry and very happy" just by reading digits top to bottom. A picture makes that instantly obvious.

A **plotting library** is a tool that turns numbers into a picture — bars, dots, lines — drawn on screen. It matters because human eyes are extremely good at spotting shape, outliers, and gaps in a picture, and quite bad at spotting the same things in a column of digits. Tiny example: the twelve text lengths `59, 39, 43, 39, 46, 38, 38, 37, 47, 40, 37, 30` are hard to summarize by eye as a list — but drawn as a histogram (Step 2), the shape "most tickets cluster in the high-30s to high-40s, with one much longer ticket off to the side" jumps out in under a second. The library used throughout this course is **matplotlib**, the standard Python plotting library; its most common interface is called `pyplot`, always imported under the nickname `plt`. Calling `plt.show()` at the end of a cell displays whatever was built up by the calls before it — this is the conventional, explicit way to display a plot, and works the same way across different notebook setups.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

tickets = {
    "ticket_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "text": [
        "My payment failed twice and I need this resolved right now!",
        "Do you have this item in a larger size?",
        "Where is my order, it has not arrived yet!!",
        "Thanks for the quick delivery, love it!",
        "The app keeps crashing when I try to check out",
        "Can I get a refund for a damaged item?",
        "Why was I charged twice for one order?",
        "My package arrived completely crushed",
        "Great quality, exactly as described, thank you!",
        "I need an invoice copy for my accountant",
        "Can you cancel my subscription please",
        "I was double billed this month",
    ],
    "category": [
        "billing", "product", "shipping", "shipping", "product", "billing",
        "billing", "shipping", "product", "billing", "billing", "billing",
    ],
    "satisfaction_rating": [2, 4, 1, 5, None, 3, 2, 1, 5, 4, 3, 2],
}

df = pd.DataFrame(tickets)
df["length"] = df["text"].str.len()
df["exclamations"] = df["text"].str.count("!")

print(df[["ticket_id", "category", "satisfaction_rating", "length"]])

ModuleNotFoundError: No module named 'pandas'

## Step 2 — The histogram: one column's distribution

**Plain definition:** a **histogram** groups a column of numbers into equal-width buckets (called **bins**) and draws a bar for each bucket, as tall as the count of values that landed in it.

**Why it matters:** it answers "what does this column's spread look like" in one glance — are the values bunched together, spread evenly, or is there one value way off from the rest (an **outlier** — a value far outside where the rest of the data sits)?

**Tiny concrete example:** the `length` column holds `59, 39, 43, 39, 46, 38, 38, 37, 47, 40, 37, 30`. Sorted, that is `30, 37, 37, 38, 38, 39, 39, 40, 43, 46, 47, 59` — eleven of the twelve values sit between 30 and 47, and one, `59`, sits well past the rest. A histogram turns that into a picture: a tall cluster of bars in the middle of the range, one lonely short bar out to the right for the `59`, with nothing in between — the outlier is visible in the empty gap, not just the tall bars.

`plt.hist(data, bins=n)` builds this automatically: it picks `n` equal-width buckets spanning the data's min to max, counts how many values fall in each, and draws the bars.

In [ ]:
plt.hist(df["length"], bins=5, edgecolor="black")
plt.title("Distribution of ticket text length")
plt.xlabel("Length (characters)")
plt.ylabel("Number of tickets")
plt.show()

## Step 3 — The bar chart: counting a category

**Plain definition:** a **bar chart** draws one bar per category, as tall as how many rows belong to that category.

**Why it matters:** it is the fastest way to spot **class imbalance** — when one category has far more examples than another. This matters enormously later on: Phase 2's first classifier will learn to predict ticket category, and a model trained mostly on billing tickets can get lazy and just guess "billing" most of the time, scoring deceptively well on accuracy while barely learning `product` or `shipping` at all (Lesson 21 covers this properly). Catching the imbalance now, with a bar chart, is exactly the kind of thing a printed table of category names makes easy to miss.

**Tiny concrete example:** `df['category'].value_counts()` counts how many rows have each category value: `billing` appears 6 times, `product` 3 times, `shipping` 3 times, out of 12 tickets total — billing alone is half the data. A bar chart makes that 6-vs-3-vs-3 gap visually obvious in a way that scanning twelve category names in a printed table does not.

(Note: tied counts — here `product` and `shipping` both have 3 — may print in either order depending on the pandas version; this is a minor display detail, not a difference in the underlying counts.)

In [ ]:
category_counts = df["category"].value_counts()
print(category_counts)

category_counts.plot(kind="bar")
plt.title("Ticket count by category")
plt.xlabel("Category")
plt.ylabel("Number of tickets")
plt.show()

## Step 4 — The scatter plot: does one number relate to another?

First, a callback to Lesson 06: `satisfaction_rating` still has one missing value (ticket 5, never rated). A scatter plot needs a real number in both spots for every point it draws, so the missing row is dropped first with `.dropna(subset=['satisfaction_rating'])` — the exact tool from last lesson, now used because a later step actually needs it, not just as an exercise.

**Plain definition:** a **scatter plot** draws one dot per row, placing it left-to-right by one number and up-down by a second number.

**Why it matters:** it answers "as one number goes up, does the other tend to go up, go down, or do nothing in particular?" — a relationship (or the honest lack of one) that a table of paired numbers does not reveal at a glance.

**Tiny concrete example:** plotting `length` (x) against `satisfaction_rating` (y) for the 11 rated tickets gives points like `(59, 2)`, `(47, 5)`, `(30, 2)`, `(38, 3)` — a long, 59-character ticket got a low rating (2), but so did a short, 30-character one, while a fairly long 47-character ticket got the top rating (5). The dots land all over the plot with no visible upward or downward drift. That is a real, useful finding: it means ticket length and satisfaction do **not** appear related in this data — worth knowing *before* spending time building a model that assumes they are, which a scatter plot reveals in one picture instead of after the fact.

In [ ]:
df_clean = df.dropna(subset=["satisfaction_rating"])

plt.scatter(df_clean["length"], df_clean["satisfaction_rating"])
plt.title("Ticket length vs. satisfaction rating")
plt.xlabel("Length (characters)")
plt.ylabel("Satisfaction rating (1-5)")
plt.show()

## Step 5 — The box plot: comparing groups at once

**Plain definition:** a **box plot** summarizes a column's spread with one small picture per group: a box covering the middle half of the values, a line inside it marking the **median** (the middle value when all values are sorted), and thin "whisker" lines reaching out to the lowest and highest values.

**Why it matters:** it answers "is this number's spread different across categories" in a single picture — instead of computing min, max, and median by hand for every group separately.

**Tiny concrete example:** grouping the 11 rated tickets' `satisfaction_rating` by category: `billing` has ratings `2, 3, 2, 4, 3, 2` (clustered low-to-middle, median 2.5), `product` has `4, 5` (clustered high, median 4.5), and `shipping` has `1, 5, 1` (one 5 next to two 1s — the widest possible spread for this data, median 1). A box plot draws all three side by side: billing's box sits low and narrow, product's sits high and narrow, and shipping's whiskers stretch the full length of the scale — visually flagging shipping as the category with the least consistent customer experience, a pattern the raw numbers hide unless someone sorts and compares each group by hand.

`df.boxplot(column=..., by=...)` builds all of this directly from the DataFrame; pandas adds its own default title text above the plot, which `plt.suptitle("")` clears away so only our own `plt.title(...)` remains (a small, well-known pandas plotting quirk, worth confirming still applies against the installed pandas version).

In [ ]:
df_clean.boxplot(column="satisfaction_rating", by="category")
plt.suptitle("")
plt.title("Satisfaction rating by category")
plt.xlabel("Category")
plt.ylabel("Satisfaction rating (1-5)")
plt.show()

## Recap: four plots, four different questions

- **Histogram** — what does one column's spread look like, and are there outliers? (One long ticket stood apart from the rest.)
- **Bar chart** — are the categories balanced? (Billing had twice as many tickets as product or shipping.)
- **Scatter plot** — do two numbers move together? (Length and rating did not appear related.)
- **Box plot** — does a column's spread differ across groups? (Shipping ratings swung the widest, from very unhappy to very happy.)

Every one of these caught something a printed table of the same 12 rows would not have made obvious at a glance — that is the entire case for looking before modeling.